In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
df = pd.read_csv('../data/train_data.csv')
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32427 entries, 0 to 32426
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   price_total        32427 non-null  int64  
 1   area               32427 non-null  float64
 2   price_per_m2       32427 non-null  float64
 3   category           32427 non-null  object 
 4   latitude           32427 non-null  float64
 5   longitude          32427 non-null  float64
 6   district           32427 non-null  object 
 7   province           32427 non-null  object 
 8   legal_status       32427 non-null  object 
 9   frontage           32427 non-null  float64
 10  road_width         32427 non-null  float64
 11  num_bedrooms       32427 non-null  int64  
 12  num_toilets        32427 non-null  int64  
 13  num_floors         32427 non-null  int64  
 14  num_schools_1km    32427 non-null  int64  
 15  num_hospitals_2km  32427 non-null  int64  
 16  num_markets_1km    324

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Lựa chọn các biến độc lập (X) và biến phụ thuộc (y)
# Loại bỏ cột 'price_total', 'price_billion' và 'price_per_m2' khỏi tập feature
X = df.drop(columns=['price_total', 'price_billion', 'price_per_m2'])
y = df['price_billion'] # Dự đoán giá theo đơn vị Tỷ VNĐ

# Chia dữ liệu thành tập Train (80%) và Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Tự động nhận diện cột dạng số và dạng phân loại (category)
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Tạo Pipeline xử lý số: Điền giá trị thiếu (median) và Chuẩn hóa dữ liệu (StandardScaler)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Tạo Pipeline xử lý phân loại: Điền giá trị thiếu (mode) và One-Hot Encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Gộp các Pipeline lại
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Đã thiết lập xong quy trình tiền xử lý dữ liệu!")

Đã thiết lập xong quy trình tiền xử lý dữ liệu!


In [5]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

name = "CatBoost"
model = CatBoostRegressor(random_state=42, verbose=0)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

# Huấn luyện
pipeline.fit(X_train, y_train)

# Dự đoán
y_pred = pipeline.predict(X_test)

# Đánh giá
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"✔️ Đã train xong: {name}")
print(f"R-square (R2): {r2}")
print(f"RMS (Tỷ VNĐ): {rmse}")
print(f"MAE (Tỷ VNĐ): {mae}")

✔️ Đã train xong: CatBoost
R-square (R2): 0.8004315967661961
RMS (Tỷ VNĐ): 9.434915319149416
MAE (Tỷ VNĐ): 3.9721118889574165


In [6]:
import joblib

# 1. Lưu toàn bộ pipeline (bao gồm cả preprocessor và model)
joblib.dump(pipeline, 'catboost_pipeline.joblib')
print("Đã lưu toàn bộ Pipeline thành công vào 'catboost_pipeline.joblib'!")

# --- Sau này ở một file khác, bạn load lên để dự đoán như sau ---
# loaded_pipeline = joblib.load('catboost_pipeline.joblib')
# new_predictions = loaded_pipeline.predict(X_new)

Đã lưu toàn bộ Pipeline thành công vào 'catboost_pipeline.joblib'!


In [7]:
# 1. Trích xuất model CatBoost từ Pipeline và lưu thành file .cbm
catboost_model = pipeline.named_steps['model']
catboost_model.save_model("catboost_model.cbm")
print("Đã lưu model CatBoost thành công vào 'catboost_model.cbm'!")

# 2. Lưu bộ preprocessor ra một file riêng để sau này dùng
preprocessor_only = pipeline.named_steps['preprocessor']
joblib.dump(preprocessor_only, 'preprocessor.joblib')

# --- Sau này khi load lại để dự đoán ---
# from catboost import CatBoostRegressor
# loaded_preprocessor = joblib.load('preprocessor.joblib')
# loaded_model = CatBoostRegressor()
# loaded_model.load_model("catboost_model.cbm")
#
# Dữ liệu mới vào phải qua preprocessor trước:
# X_new_transformed = loaded_preprocessor.transform(X_new)
# predictions = loaded_model.predict(X_new_transformed)

Đã lưu model CatBoost thành công vào 'catboost_model.cbm'!


['preprocessor.joblib']